# AeroTwin AI  
## Asistente experto con Gemini, RAG, ChromaDB y LangGraph

**Tarea 7 – Máster Big Data e Inteligencia Artificial**

Este notebook implementa un MVP académico de **AeroTwin AI**, un asistente experto sobre aeropuertos de Aena que responde preguntas utilizando una base de conocimiento oficial compuesta por documentos de **Aena** y **ENAIRE/AIP**.

El proyecto incluye:

- Carga de documentos PDF y HTML.
- Limpieza y división en chunks.
- Embeddings con Gemini.
- Base vectorial con ChromaDB.
- Recuperación semántica mediante RAG.
- System prompt personalizado.
- Agente con LangGraph.
- Memoria básica de conversación.
- Celda de chat interactiva.
- Preguntas documentadas para la demo.

## 0. Configuración inicial

Este apartado prepara el entorno para que el notebook pueda ejecutarse tanto en **Google Colab** como en local/VS Code.

Si el notebook se abre en Colab directamente desde GitHub, es posible que el repositorio completo no esté cargado en `/content`. Por eso, esta celda comprueba si existe `data/raw/` y, si hace falta, clona el repositorio.

In [1]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/mg15best/aerotwin-ai.git"
REPO_NAME = "aerotwin-ai"


def find_repo_root(start: Path) -> Path | None:
    """
    Busca hacia arriba una carpeta que parezca la raíz del repositorio.
    """
    start = start.resolve()
    candidates = [start] + list(start.parents)

    for candidate in candidates:
        if (candidate / "requirements.txt").exists() and (candidate / "data" / "raw").exists():
            return candidate

    return None


current_root = find_repo_root(Path.cwd())

if current_root is not None:
    os.chdir(current_root)
else:
    # Caso habitual en Google Colab: el notebook está abierto, pero el repo completo no está clonado.
    if Path("/content").exists():
        repo_dir = Path("/content") / REPO_NAME

        if not repo_dir.exists():
            print("Clonando repositorio en Colab...")
            subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)
        else:
            print("El repositorio ya existe en /content.")

        os.chdir(repo_dir)
    else:
        raise FileNotFoundError(
            "No se ha encontrado la raíz del repositorio. "
            "Abre este notebook desde la carpeta del proyecto o clona el repositorio."
        )

print("Carpeta actual:")
print(Path.cwd())

print("\nContenido principal:")
for item in sorted(Path(".").iterdir()):
    print("-", item)

print("\n¿Existe requirements.txt?", Path("requirements.txt").exists())
print("¿Existe data/raw?", Path("data/raw").exists())

Clonando repositorio en Colab...
Carpeta actual:
/content/aerotwin-ai

Contenido principal:
- .env.example
- .git
- .gitignore
- README.md
- data
- notebooks
- outputs
- requirements.txt
- src

¿Existe requirements.txt? True
¿Existe data/raw? True


### 0.1. Instalación de dependencias

Ejecuta esta celda una vez al iniciar el entorno. En Colab puede tardar unos minutos.

In [2]:
import subprocess
import sys
from pathlib import Path

if not Path("requirements.txt").exists():
    raise FileNotFoundError("No se encuentra requirements.txt. Comprueba que estás en la raíz del repositorio.")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True
)

print("Dependencias instaladas correctamente.")

Dependencias instaladas correctamente.


### 0.2. Configuración de la API key de Gemini

La clave real debe introducirse en el entorno de ejecución, no en GitHub.

Opciones:

1. Crear un archivo `.env` local con `GOOGLE_API_KEY=...`.
2. Introducir la clave manualmente cuando el notebook la solicite.

El archivo `.env` nunca debe subirse al repositorio.

In [3]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Pega aquí tu Google Gemini API Key: ")

GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")
GEMINI_EMBEDDING_MODEL = os.getenv("GEMINI_EMBEDDING_MODEL", "gemini-embedding-2-preview")
GEMINI_EMBEDDING_DIMENSIONS = int(os.getenv("GEMINI_EMBEDDING_DIMENSIONS", "768"))

print("API key configurada:", bool(os.environ.get("GOOGLE_API_KEY")))
print("Modelo Gemini:", GEMINI_MODEL)
print("Modelo de embeddings:", GEMINI_EMBEDDING_MODEL)
print("Dimensiones embeddings:", GEMINI_EMBEDDING_DIMENSIONS)

Pega aquí tu Google Gemini API Key: ··········
API key configurada: True
Modelo Gemini: gemini-2.5-flash
Modelo de embeddings: gemini-embedding-2-preview
Dimensiones embeddings: 768


### 0.3. Prueba rápida de Gemini

Antes de construir el RAG, comprobamos que el modelo de lenguaje responde correctamente.

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model=GEMINI_MODEL,
    temperature=0
)

response = llm.invoke("Responde solo con esta frase: Gemini funciona correctamente.")
print(response.content)

Gemini funciona correctamente.


### 0.4. Prueba rápida de Gemini Embeddings

Esta prueba comprueba que el modelo de embeddings genera vectores correctamente.

In [5]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model=GEMINI_EMBEDDING_MODEL,
    output_dimensionality=GEMINI_EMBEDDING_DIMENSIONS
)

test_vector = embeddings.embed_query("AeroTwin AI test embedding")

print("Embedding generado correctamente.")
print("Dimensión del vector:", len(test_vector))
print("Primeros 5 valores:", test_vector[:5])

Embedding generado correctamente.
Dimensión del vector: 768
Primeros 5 valores: [-0.037928145, 0.00288188, -0.017322036, -0.011273562, -0.06233103]


## 1. Carga y preparación de documentos

En este apartado se cargan los documentos oficiales que forman la base de conocimiento de AeroTwin AI.

El proyecto utiliza documentos en formato PDF y HTML procedentes de Aena y ENAIRE/AIP. Se excluye por defecto la carpeta `support_optional/` para mantener el RAG principal centrado en las fuentes más relevantes.

Cada documento se carga con metadatos para identificar:

- Archivo de origen.
- Carpeta de origen.
- Tipo de documento.
- Público objetivo principal.

In [6]:
from pathlib import Path
from collections import Counter

RAW_DATA_DIR = Path("data/raw")
EXCLUDED_FOLDERS = ["support_optional"]
SUPPORTED_EXTENSIONS = [".pdf", ".html", ".htm"]

document_files = []

for path in RAW_DATA_DIR.rglob("*"):
    if not path.is_file():
        continue

    if path.name == ".gitkeep":
        continue

    if any(excluded in path.parts for excluded in EXCLUDED_FOLDERS):
        continue

    if path.suffix.lower() in SUPPORTED_EXTENSIONS:
        document_files.append(path)

document_files = sorted(document_files)

print("Documentos principales encontrados para el RAG:", len(document_files))
print()

for file in document_files:
    print("-", file)

print("\nResumen por carpeta:")
folder_counter = Counter(str(file.parent) for file in document_files)

for folder, count in folder_counter.items():
    print(f"{folder}: {count} documentos")

Documentos principales encontrados para el RAG: 18

- data/raw/aena_airlines/aena_airlines_commercial_incentives.html
- data/raw/aena_airlines/aena_airlines_marketing_support.html
- data/raw/aena_airlines/aena_airlines_new_routes.html
- data/raw/aena_airlines/aena_airlines_portal.html
- data/raw/aena_airlines/aena_airports_destinations_network.html
- data/raw/aena_airlines/aena_mad_business_profile.html
- data/raw/aena_airlines/aena_price_guide_july_2026.pdf
- data/raw/aena_passengers/aena_mad_airport_guide_maps.html
- data/raw/aena_passengers/aena_mad_passenger_homepage.html
- data/raw/aena_passengers/aena_passengers_general.html
- data/raw/aena_passengers/aena_passengers_prm_general.html
- data/raw/aena_passengers/aena_tfs_information_meeting_points.html
- data/raw/aena_passengers/aena_tfs_special_needs.html
- data/raw/enaire_aip/enaire_aip_gcts_tenerife_sur.pdf
- data/raw/enaire_aip/enaire_aip_gcxo_tenerife_norte.pdf
- data/raw/enaire_aip/enaire_aip_lebl_barcelona.html
- data/raw/en

In [7]:
from bs4 import BeautifulSoup
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader


def infer_audience_from_path(path: Path) -> str:
    """
    Infers the main audience of a document from its folder.
    """
    path_parts = set(path.parts)

    if "aena_passengers" in path_parts:
        return "passenger"

    if "aena_airlines" in path_parts:
        return "travel_industry"

    if "enaire_aip" in path_parts:
        return "aviation_professional"

    return "general"


def infer_document_group_from_path(path: Path) -> str:
    """
    Creates a human-readable document group from the source folder.
    """
    if "aena_passengers" in path.parts:
        return "Aena passenger information"

    if "aena_airlines" in path.parts:
        return "Aena airline and route-development information"

    if "enaire_aip" in path.parts:
        return "ENAIRE / AIP aeronautical information"

    return "General source"


def clean_text(text: str) -> str:
    """
    Basic text cleaning to remove excessive blank lines and spaces.
    """
    lines = [line.strip() for line in text.splitlines()]
    lines = [line for line in lines if line]
    return "\n".join(lines)


def load_html_document(path: Path) -> Document:
    """
    Loads and cleans a local HTML document.
    """
    html = path.read_text(encoding="utf-8", errors="ignore")
    soup = BeautifulSoup(html, "lxml")

    for tag in soup(["script", "style", "nav", "footer", "header"]):
        tag.decompose()

    text = clean_text(soup.get_text(separator="\n"))

    return Document(
        page_content=text,
        metadata={
            "source": str(path),
            "source_file": path.name,
            "source_folder": path.parent.name,
            "document_group": infer_document_group_from_path(path),
            "file_type": "html",
            "audience": infer_audience_from_path(path),
        },
    )


def load_pdf_documents(path: Path) -> list[Document]:
    """
    Loads a local PDF document page by page.
    """
    loader = PyPDFLoader(str(path))
    pages = loader.load()

    cleaned_pages = []

    for page_number, page in enumerate(pages, start=1):
        page.page_content = clean_text(page.page_content)

        if not page.page_content.strip():
            continue

        page.metadata.update(
            {
                "source": str(path),
                "source_file": path.name,
                "source_folder": path.parent.name,
                "document_group": infer_document_group_from_path(path),
                "file_type": "pdf",
                "audience": infer_audience_from_path(path),
                "page": page_number,
            }
        )

        cleaned_pages.append(page)

    return cleaned_pages


loaded_documents = []

for file_path in document_files:
    if file_path.suffix.lower() == ".pdf":
        loaded_documents.extend(load_pdf_documents(file_path))

    elif file_path.suffix.lower() in [".html", ".htm"]:
        loaded_documents.append(load_html_document(file_path))

print("Documentos cargados en LangChain:", len(loaded_documents))

print("\nResumen por audiencia:")
audience_counter = Counter(doc.metadata.get("audience", "unknown") for doc in loaded_documents)

for audience, count in audience_counter.items():
    print(f"- {audience}: {count}")

print("\nEjemplo de metadatos del primer documento:")
print(loaded_documents[0].metadata)

print("\nVista previa del contenido:")
print(loaded_documents[0].page_content[:1000])

/tmp/ipykernel_898/659374061.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Documentos cargados en LangChain: 160

Resumen por audiencia:
- travel_industry: 117
- passenger: 6
- aviation_professional: 37

Ejemplo de metadatos del primer documento:
{'source': 'data/raw/aena_airlines/aena_airlines_commercial_incentives.html', 'source_file': 'aena_airlines_commercial_incentives.html', 'source_folder': 'aena_airlines', 'document_group': 'Aena airline and route-development information', 'file_type': 'html', 'audience': 'travel_industry'}

Vista previa del contenido:
Commercial Incentives | Summer and Winter Season | Aena
Incentives
Aena commercial incentives
breadcrumb-header
Discover the incentives and discounts Aena offers airlines
Incentive for airports with more than 3,5 million passengers
Apply before the end of the season that entitles the generation of the incentive
Incentive for airports with more than 3,5 million passengers (PDF - 339.71 kB)
Incentive on routes to Asia
Apply before the end of the season that entitles the generation of the incentive
Incenti

## 2. División de documentos en chunks

Una vez cargados los documentos, se dividen en fragmentos más pequeños llamados **chunks**.

Este paso es necesario porque los modelos de embeddings y el sistema RAG funcionan mejor cuando los textos se dividen en unidades de información manejables.

Se utiliza `RecursiveCharacterTextSplitter`, manteniendo cierto solapamiento entre fragmentos para no perder contexto entre partes consecutivas del documento.

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from collections import Counter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(loaded_documents)

# Eliminar posibles chunks vacíos
chunks = [chunk for chunk in chunks if chunk.page_content.strip()]

# Añadir metadatos de control
for index, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = index
    chunk.metadata["chunk_length"] = len(chunk.page_content)

print("Número total de chunks generados:", len(chunks))

print("\nChunks por audiencia:")
audience_counter = Counter(chunk.metadata.get("audience", "unknown") for chunk in chunks)

for audience, count in audience_counter.items():
    print(f"- {audience}: {count}")

print("\nChunks por carpeta:")
folder_counter = Counter(chunk.metadata.get("source_folder", "unknown") for chunk in chunks)

for folder, count in folder_counter.items():
    print(f"- {folder}: {count}")

print("\nEjemplo de chunk:")
print("Metadatos:", chunks[0].metadata)
print("\nContenido:")
print(chunks[0].page_content[:1000])

Número total de chunks generados: 946

Chunks por audiencia:
- travel_industry: 291
- passenger: 19
- aviation_professional: 636

Chunks por carpeta:
- aena_airlines: 291
- aena_passengers: 19
- enaire_aip: 636

Ejemplo de chunk:
Metadatos: {'source': 'data/raw/aena_airlines/aena_airlines_commercial_incentives.html', 'source_file': 'aena_airlines_commercial_incentives.html', 'source_folder': 'aena_airlines', 'document_group': 'Aena airline and route-development information', 'file_type': 'html', 'audience': 'travel_industry', 'chunk_id': 0, 'chunk_length': 924}

Contenido:
Commercial Incentives | Summer and Winter Season | Aena
Incentives
Aena commercial incentives
breadcrumb-header
Discover the incentives and discounts Aena offers airlines
Incentive for airports with more than 3,5 million passengers
Apply before the end of the season that entitles the generation of the incentive
Incentive for airports with more than 3,5 million passengers (PDF - 339.71 kB)
Incentive on routes to Asia


## 3. Selección de chunks para el MVP

En una ejecución completa se podrían indexar todos los chunks. Sin embargo, en el nivel gratuito de la API de Gemini puede haber límites temporales de uso.

Para que la demo sea estable, este notebook usa por defecto un modo **MVP/demo**, que selecciona una muestra equilibrada de chunks por documento.

Esto permite demostrar correctamente:

- Carga documental.
- Embeddings con Gemini.
- ChromaDB.
- RAG.
- LangGraph.
- Memoria.
- Preguntas de demo.

Si se desea indexar todo el corpus, se puede cambiar `INDEX_MODE = "full"`.

In [9]:
from collections import defaultdict

INDEX_MODE = "demo"  # Opciones: "demo" o "full"
MAX_CHUNKS_PER_FILE = 3

if INDEX_MODE not in ["demo", "full"]:
    raise ValueError("INDEX_MODE debe ser 'demo' o 'full'.")

if INDEX_MODE == "full":
    chunks_for_index = chunks
else:
    chunks_by_file = defaultdict(list)

    for chunk in chunks:
        source_file = chunk.metadata.get("source_file", "unknown")
        chunks_by_file[source_file].append(chunk)

    chunks_for_index = []

    for source_file, file_chunks in sorted(chunks_by_file.items()):
        selected_chunks = file_chunks[:MAX_CHUNKS_PER_FILE]
        chunks_for_index.extend(selected_chunks)
        print(f"{source_file}: {len(selected_chunks)} chunks seleccionados")

print("\nModo de indexación:", INDEX_MODE)
print("Total de chunks disponibles:", len(chunks))
print("Total de chunks que se indexarán:", len(chunks_for_index))

aena_airlines_commercial_incentives.html: 3 chunks seleccionados
aena_airlines_marketing_support.html: 3 chunks seleccionados
aena_airlines_new_routes.html: 2 chunks seleccionados
aena_airlines_portal.html: 3 chunks seleccionados
aena_airports_destinations_network.html: 3 chunks seleccionados
aena_mad_airport_guide_maps.html: 1 chunks seleccionados
aena_mad_business_profile.html: 3 chunks seleccionados
aena_mad_passenger_homepage.html: 3 chunks seleccionados
aena_passengers_general.html: 3 chunks seleccionados
aena_passengers_prm_general.html: 3 chunks seleccionados
aena_price_guide_july_2026.pdf: 3 chunks seleccionados
aena_tfs_information_meeting_points.html: 3 chunks seleccionados
aena_tfs_special_needs.html: 3 chunks seleccionados
enaire_aip_gcts_tenerife_sur.pdf: 3 chunks seleccionados
enaire_aip_gcxo_tenerife_norte.pdf: 3 chunks seleccionados
enaire_aip_lebl_barcelona.html: 3 chunks seleccionados
enaire_aip_lemd_madrid_barajas.html: 3 chunks seleccionados
enaire_aip_spain_general

## 4. Creación de la base vectorial con ChromaDB

En este apartado se crea la base de datos vectorial del proyecto.

Cada chunk seleccionado se transforma en un embedding mediante Gemini Embeddings. Después, esos vectores se almacenan en ChromaDB para permitir búsquedas semánticas.

La carpeta `chroma_db/` se genera automáticamente al ejecutar el notebook y no se sube al repositorio, ya que puede regenerarse a partir de los documentos originales.

In [10]:
import shutil
from pathlib import Path

from langchain_chroma import Chroma

CHROMA_DIR = Path("chroma_db")
COLLECTION_NAME = "aerotwin_ai"

RESET_CHROMA = True

if RESET_CHROMA and CHROMA_DIR.exists():
    shutil.rmtree(CHROMA_DIR)

try:
    vectorstore = Chroma.from_documents(
        documents=chunks_for_index,
        embedding=embeddings,
        persist_directory=str(CHROMA_DIR),
        collection_name=COLLECTION_NAME
    )

    print("Base vectorial creada correctamente.")
    print("Directorio de ChromaDB:", CHROMA_DIR)
    print("Colección:", COLLECTION_NAME)
    print("Número de vectores almacenados:", vectorstore._collection.count())

except Exception as error:
    print("No se ha podido crear la base vectorial.")
    print("\nSugerencia:")
    print("- Espera unos minutos si aparece un error de cuota o RESOURCE_EXHAUSTED.")
    print("- Reduce MAX_CHUNKS_PER_FILE a 1 o 2 si estás usando el nivel gratuito.")
    print("- Vuelve a ejecutar desde el Apartado 3.")
    raise error

Base vectorial creada correctamente.
Directorio de ChromaDB: chroma_db
Colección: aerotwin_ai
Número de vectores almacenados: 51


## 5. Prueba del retriever semántico

Antes de conectar el agente, verificamos que ChromaDB recupera fragmentos relevantes de la base de conocimiento.

In [11]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

test_query = "¿Qué apoyo ofrece Aena a las aerolíneas que quieren abrir nuevas rutas?"

retrieved_docs = retriever.invoke(test_query)

print("Consulta de prueba:")
print(test_query)

print("\nDocumentos recuperados:", len(retrieved_docs))

for i, doc in enumerate(retrieved_docs, start=1):
    print("\n" + "=" * 80)
    print(f"Resultado {i}")
    print("Fuente:", doc.metadata.get("source_file"))
    print("Carpeta:", doc.metadata.get("source_folder"))
    print("Audiencia:", doc.metadata.get("audience"))
    print("Página:", doc.metadata.get("page", "N/A"))
    print("-" * 80)
    print(doc.page_content[:800])

Consulta de prueba:
¿Qué apoyo ofrece Aena a las aerolíneas que quieren abrir nuevas rutas?

Documentos recuperados: 5

Resultado 1
Fuente: aena_airlines_marketing_support.html
Carpeta: aena_airlines
Audiencia: travel_industry
Página: N/A
--------------------------------------------------------------------------------
Marketing Support Actions for Airlines | Aena
Marketing support
Marketing support for the promotion and development of new air routes
breadcrumb-header
Dedicated area at the airport for the first flight ceremony.
Special treatment for passengers of the first flight.
Support for airlines related to communication with the relevant authorities and local press for first flight ceremonies and/or promotion of new air routes
More detailed information on marketing support measures is available at the following link:
Description of Aena's marketing support actions. Summer 2026.pdf
Advertising in the airport
Free advertising spaces (MUPI) and digital media in Aena airports.
Online:

## 6. System prompt personalizado

El system prompt define el comportamiento del asistente. En este proyecto, AeroTwin AI tiene dos voces principales:

- **Passenger Voice**: lenguaje claro, práctico y orientado al pasajero.
- **Travel Industry Voice**: lenguaje profesional, orientado a aerolíneas, rutas, turismo y negocio aeroportuario.

También se incluye una regla clave del RAG: el asistente debe responder usando solo el contexto recuperado y debe reconocer sus límites cuando la base documental no contiene suficiente información.

In [12]:
SYSTEM_PROMPT = """
Eres AeroTwin AI, un asistente experto en aeropuertos de Aena construido para un proyecto académico de Inteligencia Artificial Generativa.

Tu función es responder preguntas utilizando una base de conocimiento oficial compuesta por documentos de Aena y ENAIRE/AIP.

Tienes dos modos principales de respuesta:

1. Passenger Voice:
   - Para pasajeros y usuarios generales del aeropuerto.
   - Usa un lenguaje claro, directo y útil.
   - Prioriza orientación práctica, pasos concretos y explicaciones sencillas.
   - Evita tecnicismos innecesarios.

2. Travel Industry Voice:
   - Para aerolíneas, agencias de viaje, turoperadores, equipos de desarrollo de rutas y profesionales turísticos.
   - Usa un tono profesional y analítico.
   - Prioriza información sobre rutas, incentivos, conectividad, operación, mercado y oportunidades de negocio.
   - Puede utilizar lenguaje sectorial, pero siempre de forma comprensible.

Reglas de respuesta:

- Responde únicamente con la información disponible en el contexto recuperado.
- Si el contexto no contiene información suficiente, dilo claramente.
- No inventes horarios, rutas, tarifas, procedimientos, normativa o datos operativos.
- No proporciones información de vuelos en tiempo real ni alertas operativas en directo.
- Si una pregunta requiere datos en tiempo real, explica que el MVP no dispone de esa información.
- Cuando sea útil, diferencia entre información para pasajeros e información para profesionales.
- Mantén la coherencia con la conversación previa.
- Responde en el mismo idioma de la pregunta del usuario, salvo que se solicite otro idioma.
- Al final de la respuesta, incluye una breve sección "Fuentes consultadas" con los nombres de los documentos recuperados.
"""

print(SYSTEM_PROMPT[:1500])


Eres AeroTwin AI, un asistente experto en aeropuertos de Aena construido para un proyecto académico de Inteligencia Artificial Generativa.

Tu función es responder preguntas utilizando una base de conocimiento oficial compuesta por documentos de Aena y ENAIRE/AIP.

Tienes dos modos principales de respuesta:

1. Passenger Voice:
   - Para pasajeros y usuarios generales del aeropuerto.
   - Usa un lenguaje claro, directo y útil.
   - Prioriza orientación práctica, pasos concretos y explicaciones sencillas.
   - Evita tecnicismos innecesarios.

2. Travel Industry Voice:
   - Para aerolíneas, agencias de viaje, turoperadores, equipos de desarrollo de rutas y profesionales turísticos.
   - Usa un tono profesional y analítico.
   - Prioriza información sobre rutas, incentivos, conectividad, operación, mercado y oportunidades de negocio.
   - Puede utilizar lenguaje sectorial, pero siempre de forma comprensible.

Reglas de respuesta:

- Responde únicamente con la información disponible en el

### 6.1. Justificación académica del prompt

Este prompt está diseñado para cumplir los requisitos del ejercicio porque:

- Define un rol experto claro.
- Delimita el dominio del asistente.
- Diferencia dos públicos objetivos.
- Reduce alucinaciones obligando a responder desde el contexto recuperado.
- Explicita los límites del MVP académico.
- Mantiene coherencia conversacional para utilizar memoria.
- Favorece respuestas útiles y demostrables durante la presentación.

## 7. Construcción del agente con LangGraph

En este apartado se construye el agente RAG usando LangGraph.

El grafo tiene dos nodos principales:

1. `retrieve_context`: recupera documentos relevantes desde ChromaDB.
2. `generate_answer`: genera la respuesta final con Gemini usando el contexto recuperado.

Además, se incorpora `MemorySaver` para mantener memoria básica de conversación por `thread_id`.

In [13]:
from typing import Annotated, TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage


class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    audience: str
    airport: str
    context: str
    sources: list[str]


def format_retrieved_context(docs) -> tuple[str, list[str]]:
    """
    Converts retrieved documents into a formatted context block and source list.
    """
    context_blocks = []
    sources = []

    for i, doc in enumerate(docs, start=1):
        source_file = doc.metadata.get("source_file", "unknown")
        page = doc.metadata.get("page", "N/A")
        audience = doc.metadata.get("audience", "unknown")
        document_group = doc.metadata.get("document_group", "unknown")

        sources.append(source_file)

        block = f"""
[Documento {i}]
Fuente: {source_file}
Página: {page}
Audiencia: {audience}
Grupo documental: {document_group}

Contenido:
{doc.page_content}
"""
        context_blocks.append(block.strip())

    unique_sources = sorted(set(sources))

    return "\n\n---\n\n".join(context_blocks), unique_sources


def get_last_user_question(messages: list[BaseMessage]) -> str:
    """
    Extracts the most recent user question from the message history.
    """
    human_messages = [message for message in messages if isinstance(message, HumanMessage)]

    if not human_messages:
        return ""

    return human_messages[-1].content


def retrieve_context(state: AgentState) -> dict:
    """
    LangGraph node: retrieves relevant context from ChromaDB.
    """
    question = get_last_user_question(state["messages"])

    retrieved = retriever.invoke(question)
    context, sources = format_retrieved_context(retrieved)

    return {
        "context": context,
        "sources": sources
    }


def generate_answer(state: AgentState) -> dict:
    """
    LangGraph node: generates the final answer with Gemini.
    """
    audience = state.get("audience", "auto")
    airport = state.get("airport", "general")
    context = state.get("context", "")
    sources = state.get("sources", [])

    system_message = f"""
{SYSTEM_PROMPT}

Modo de respuesta solicitado: {audience}
Aeropuerto o ámbito indicado: {airport}

Contexto recuperado desde la base vectorial:
{context}

Fuentes recuperadas:
{", ".join(sources)}
"""

    messages = [
        SystemMessage(content=system_message),
        *state["messages"]
    ]

    answer = llm.invoke(messages)

    return {
        "messages": [answer]
    }


graph_builder = StateGraph(AgentState)

graph_builder.add_node("retrieve_context", retrieve_context)
graph_builder.add_node("generate_answer", generate_answer)

graph_builder.add_edge(START, "retrieve_context")
graph_builder.add_edge("retrieve_context", "generate_answer")
graph_builder.add_edge("generate_answer", END)

memory = MemorySaver()

aerotwin_agent = graph_builder.compile(checkpointer=memory)

print("Agente LangGraph creado correctamente.")

Agente LangGraph creado correctamente.


## 8. Función de consulta al asistente

Esta función permite hacer preguntas a AeroTwin AI indicando:

- Pregunta del usuario.
- Tipo de audiencia.
- Aeropuerto o ámbito.
- Identificador de conversación (`thread_id`) para mantener memoria.

In [27]:
def ask_aerotwin(
    question: str,
    audience: str = "auto",
    airport: str = "general",
    thread_id: str = "demo"
):
    """
    Sends a question to the AeroTwin AI LangGraph agent and returns
    a clean dictionary with answer, sources and raw LangGraph result.
    """
    config = {
        "configurable": {
            "thread_id": thread_id
        }
    }

    raw_result = aerotwin_agent.invoke(
        {
            "messages": [HumanMessage(content=question)],
            "audience": audience,
            "airport": airport
        },
        config=config
    )

    answer = raw_result["messages"][-1].content
    sources = raw_result.get("sources", [])

    print(answer)

    print("\n" + "-" * 80)
    print("Fuentes recuperadas por el retriever:")
    for source in sources:
        print("-", source)

    return {
        "answer": answer,
        "sources": sources,
        "raw_result": raw_result
    }

## 9. Pruebas individuales del RAG

Estas pruebas sirven para comprobar que el asistente responde en los dos modos principales.

In [15]:
# Prueba 1: Passenger Voice

result_passenger = ask_aerotwin(
    question="¿Qué información ofrece Aena a los pasajeros antes de viajar?",
    audience="passenger",
    airport="general",
    thread_id="passenger_demo"
)

Aena ofrece a los pasajeros diversas informaciones y servicios útiles antes de viajar para garantizar una buena experiencia. Aquí tienes un resumen:

*   **Servicios de asistencia:**
    *   Dispone de un servicio gratuito de asistencia para personas con movilidad reducida o discapacidad. Puedes solicitarlo a través de tu aerolínea, la web de Aena o llamando al teléfono de información (+34) 91 321 10 00. Es obligatorio solicitarlo al menos 48 horas antes de la salida de tu vuelo.
    *   También ofrece información y una insignia para personas con discapacidades ocultas, que ayuda al personal del aeropuerto a identificarlas y mejorar su experiencia.
*   **Información general para el viaje:**
    *   Detalles sobre el equipaje de mano.
    *   Información sobre la documentación necesaria para viajar.
    *   Consejos y normativas para viajar con bebés, menores y mujeres embarazadas.
    *   Información sobre la devolución del IVA.
    *   Guía para viajar con animales de forma segura.
  

In [16]:
# Prueba 2: Travel Industry Voice

result_industry = ask_aerotwin(
    question="¿Qué apoyo ofrece Aena a las aerolíneas que quieren abrir nuevas rutas?",
    audience="travel_industry",
    airport="general",
    thread_id="industry_demo"
)

Aena ofrece un amplio apoyo a las aerolíneas interesadas en abrir nuevas rutas, que se divide principalmente en acciones de marketing y soporte promocional, así como incentivos comerciales.

**Apoyo de Marketing y Promoción:**
Aena proporciona las siguientes acciones de apoyo para la promoción y desarrollo de nuevas rutas aéreas:
*   **Eventos de Primer Vuelo:**
    *   Área dedicada en el aeropuerto para la ceremonia del primer vuelo.
    *   Tratamiento especial para los pasajeros del primer vuelo.
    *   Apoyo en la comunicación con las autoridades relevantes y la prensa local para las ceremonias de primer vuelo y/o la promoción de nuevas rutas.
*   **Publicidad en el Aeropuerto:**
    *   Espacios publicitarios gratuitos (MUPI) y medios digitales en los aeropuertos de Aena.
    *   **Online:** Banner promocional en la web del aeropuerto y de información de vuelos, y promoción de la nueva ruta en los perfiles de redes sociales de Aena y otros canales de comunicación.
    *   **Offl

## 10. Prueba de memoria de conversación

Para demostrar memoria, se usa el mismo `thread_id` en varias preguntas consecutivas.

La segunda pregunta depende de la primera, por lo que el agente debe mantener el contexto conversacional.

In [17]:
# Primera pregunta de una conversación con memoria

memory_result_1 = ask_aerotwin(
    question="Hablemos del aeropuerto de Tenerife Sur desde el punto de vista de un profesional turístico.",
    audience="travel_industry",
    airport="Tenerife Sur",
    thread_id="memory_demo"
)

Desde la perspectiva de un profesional turístico, el Aeropuerto de Tenerife Sur (código ICAO: GCTS) es un punto clave de entrada a la isla.

Aquí tienes algunos datos relevantes:

*   **Identificación y Administración:** El aeropuerto es conocido oficialmente como GCTS - Tenerife Sur y está administrado por Aena.
*   **Ubicación Geográfica:** Se encuentra a 60 km al suroeste de la ciudad, un dato importante para la planificación de traslados y logística de los turoperadores y agencias de viajes.
*   **Datos Operacionales:**
    *   Coordenadas ARP: 280240N 0163421W.
    *   Elevación: 64 metros (209 pies).
    *   Temperatura de referencia: 28°C, un factor a considerar para la planificación operativa y de confort de los pasajeros.
*   **Recursos para el Turismo:** El aeropuerto cuenta con un punto de información turística gestionado por el Cabildo de Tenerife, lo que puede ser un recurso útil para profesionales que busquen información local detallada o apoyo para sus clientes. El teléf

In [18]:
# Segunda pregunta usando el mismo thread_id

memory_result_2 = ask_aerotwin(
    question="Ahora resume la oportunidad de ese aeropuerto en tres puntos clave.",
    audience="travel_industry",
    airport="Tenerife Sur",
    thread_id="memory_demo"
)

Desde la perspectiva de un profesional turístico, las oportunidades clave que ofrece el Aeropuerto de Tenerife Sur se pueden resumir en tres puntos:

1.  **Puerta de Entrada Estratégica:** Como principal aeropuerto de la isla, Tenerife Sur es el punto de acceso fundamental para el turismo internacional y nacional hacia Tenerife. Su ubicación a 60 km de la ciudad lo convierte en un nodo crucial para la planificación logística de paquetes turísticos y traslados.
2.  **Infraestructura de Apoyo al Turismo:** La presencia de un punto de información turística gestionado por el Cabildo de Tenerife dentro del aeropuerto demuestra un compromiso institucional con el sector. Esto ofrece recursos y soporte directo para los visitantes, lo cual es valioso para los profesionales que buscan información local o asistencia para sus clientes.
3.  **Capacidad Operacional y Conectividad:** La mención de contactos para aerolíneas clave (como Air Europa e Iberia) y servicios de handling consolidados (como Av

## 11. Chat interactivo

Esta celda permite interactuar con el asistente manualmente.

Es útil para la demo oral, porque permite introducir preguntas en directo sin modificar el código.

In [20]:
def interactive_chat():
    """
    Simple command-line chat for AeroTwin AI.
    """
    print("AeroTwin AI - Chat interactivo")
    print("Escribe 'salir' para terminar.")
    print()

    thread_id = input("Introduce un nombre para la conversación/thread_id: ").strip() or "interactive_demo"
    audience = input("Audiencia [auto/passenger/travel_industry/aviation_professional]: ").strip() or "auto"
    airport = input("Aeropuerto o ámbito [general/Tenerife Sur/Madrid-Barajas/etc.]: ").strip() or "general"

    while True:
        question = input("\nPregunta: ").strip()

        if question.lower() in ["salir", "exit", "quit"]:
            print("Chat finalizado.")
            break

        ask_aerotwin(
            question=question,
            audience=audience,
            airport=airport,
            thread_id=thread_id
        )


# Ejecutar manualmente si se desea:
# interactive_chat()

In [21]:
interactive_chat()

AeroTwin AI - Chat interactivo
Escribe 'salir' para terminar.

Introduce un nombre para la conversación/thread_id: demo_directo
Audiencia [auto/passenger/travel_industry/aviation_professional]: passenger
Aeropuerto o ámbito [general/Tenerife Sur/Madrid-Barajas/etc.]: Tenerife Sur

Pregunta: ¿Qué servicios de información tiene el aeropuerto de Tenerife Sur?
El aeropuerto de Tenerife Sur ofrece varios servicios de información para los pasajeros:

*   **Información del aeropuerto**: Puedes llamar al teléfono (+34) 913 211 000 para consultas y sugerencias generales.
*   **Información de aerolíneas**: Si necesitas contactar con tu aerolínea, aquí tienes algunos números:
    *   Air Europa: (+34) 922 759 244
    *   Aviapartner: (+34) 922 759 126
    *   Iberia: (+34) 922 759 391
*   **Objetos perdidos**: Si has perdido algo en las instalaciones del aeropuerto, puedes enviar un correo electrónico a tfs.objetosperdidos@aena.es. El servicio está abierto de 10:00 a 22:00. Si perdiste algo a bor

## 12. Preguntas de demostración documentadas

El enunciado solicita incluir ejemplos documentados. Estas preguntas cubren:

- Passenger Voice.
- Travel Industry Voice.
- Memoria de conversación.
- Limitaciones del MVP.

In [22]:
DEMO_QUESTIONS = [
    {
        "id": 1,
        "mode": "Passenger Voice",
        "audience": "passenger",
        "airport": "general",
        "thread_id": "demo_q1",
        "question": "¿Qué información ofrece Aena a los pasajeros antes de viajar?"
    },
    {
        "id": 2,
        "mode": "Passenger Voice",
        "audience": "passenger",
        "airport": "general",
        "thread_id": "demo_q2",
        "question": "¿Cómo puede solicitar asistencia una persona con movilidad reducida en un aeropuerto de Aena?"
    },
    {
        "id": 3,
        "mode": "Passenger Voice",
        "audience": "passenger",
        "airport": "Tenerife Sur",
        "thread_id": "demo_q3",
        "question": "¿Dónde puede encontrar un pasajero puntos de información o encuentro en el aeropuerto de Tenerife Sur?"
    },
    {
        "id": 4,
        "mode": "Travel Industry Voice",
        "audience": "travel_industry",
        "airport": "general",
        "thread_id": "demo_q4",
        "question": "¿Qué tipo de apoyo ofrece Aena a las aerolíneas que quieren abrir nuevas rutas?"
    },
    {
        "id": 5,
        "mode": "Travel Industry Voice",
        "audience": "travel_industry",
        "airport": "general",
        "thread_id": "demo_q5",
        "question": "¿Qué incentivos comerciales aparecen en la documentación de Aena para aerolíneas?"
    },
    {
        "id": 6,
        "mode": "Travel Industry Voice",
        "audience": "travel_industry",
        "airport": "Madrid-Barajas",
        "thread_id": "demo_q6",
        "question": "¿Por qué Madrid-Barajas puede ser relevante desde una perspectiva de desarrollo de rutas?"
    },
    {
        "id": 7,
        "mode": "Memory",
        "audience": "travel_industry",
        "airport": "Tenerife Sur",
        "thread_id": "demo_memory",
        "question": "Antes hemos hablado de Tenerife Sur. Ahora resume la oportunidad del aeropuerto desde el punto de vista de un profesional del sector turístico."
    }
]

for item in DEMO_QUESTIONS:
    print(f"{item['id']}. [{item['mode']}] {item['question']}")

1. [Passenger Voice] ¿Qué información ofrece Aena a los pasajeros antes de viajar?
2. [Passenger Voice] ¿Cómo puede solicitar asistencia una persona con movilidad reducida en un aeropuerto de Aena?
3. [Passenger Voice] ¿Dónde puede encontrar un pasajero puntos de información o encuentro en el aeropuerto de Tenerife Sur?
4. [Travel Industry Voice] ¿Qué tipo de apoyo ofrece Aena a las aerolíneas que quieren abrir nuevas rutas?
5. [Travel Industry Voice] ¿Qué incentivos comerciales aparecen en la documentación de Aena para aerolíneas?
6. [Travel Industry Voice] ¿Por qué Madrid-Barajas puede ser relevante desde una perspectiva de desarrollo de rutas?
7. [Memory] Antes hemos hablado de Tenerife Sur. Ahora resume la oportunidad del aeropuerto desde el punto de vista de un profesional del sector turístico.


In [28]:
demo_manual_questions = [
    {
        "question": "¿Qué información ofrece Aena a los pasajeros antes de viajar?",
        "audience": "passenger",
        "airport": "Aena network",
    },
    {
        "question": "¿Cómo puede solicitar asistencia una persona con movilidad reducida en un aeropuerto de Aena?",
        "audience": "passenger",
        "airport": "Aena network",
    },
    {
        "question": "¿Dónde puede encontrar un pasajero puntos de información o encuentro en el aeropuerto de Tenerife Sur?",
        "audience": "passenger",
        "airport": "Tenerife Sur",
    },
    {
        "question": "¿Qué tipo de apoyo ofrece Aena a las aerolíneas que quieren abrir nuevas rutas?",
        "audience": "travel_industry",
        "airport": "Aena network",
    },
    {
        "question": "¿Qué incentivos comerciales aparecen en la documentación de Aena para aerolíneas?",
        "audience": "travel_industry",
        "airport": "Aena network",
    },
]

manual_demo_results = []

for i, item in enumerate(demo_manual_questions, start=1):
    print("=" * 100)
    print(f"PREGUNTA {i}")
    print(item["question"])
    print("-" * 100)

    result = ask_aerotwin(
        question=item["question"],
        audience=item["audience"],
        airport=item["airport"],
        thread_id=f"demo_manual_{i}"
    )

    manual_demo_results.append(
        {
            "question": item["question"],
            "audience": item["audience"],
            "airport": item["airport"],
            "answer": result["answer"],
            "sources": result["sources"],
        }
    )

    print(result["answer"])
    print("\nFuentes:")
    for source in result["sources"]:
        print("-", source)

print("\nPreguntas de demo ejecutadas:", len(manual_demo_results))

PREGUNTA 1
¿Qué información ofrece Aena a los pasajeros antes de viajar?
----------------------------------------------------------------------------------------------------
Aena ofrece a los pasajeros diversas informaciones y servicios útiles antes de viajar para ayudarles a planificar su trayecto:

*   **Servicio de asistencia para personas con movilidad reducida o discapacidad**: Aena dispone de un servicio gratuito de asistencia en todos sus aeropuertos. Es obligatorio solicitarlo al menos 48 horas antes de la salida de tu vuelo, a través de tu aerolínea, la web de Aena o llamando al teléfono de información (+34) 91 321 10 00.
*   **Distintivo para discapacidades ocultas**: Para mejorar la experiencia de las personas con discapacidades ocultas, Aena ha creado un distintivo que el personal del aeropuerto puede identificar. Puedes encontrar más información y solicitarlo en la web de Aena.
*   **Información general de viaje**: Puedes encontrar orientación sobre temas como:
    *   Equ

### 12.1. Ejecución opcional de preguntas de demo

Para evitar consumir cuota innecesaria, la ejecución automática está desactivada por defecto.

Cambia `RUN_DEMO_AUTOMATICALLY = True` si quieres generar todas las respuestas de una vez.

In [23]:
RUN_DEMO_AUTOMATICALLY = False

demo_results = []

if RUN_DEMO_AUTOMATICALLY:
    for item in DEMO_QUESTIONS:
        print("\n" + "=" * 100)
        print(f"Pregunta {item['id']} - {item['mode']}")
        print(item["question"])
        print("=" * 100)

        result = ask_aerotwin(
            question=item["question"],
            audience=item["audience"],
            airport=item["airport"],
            thread_id=item["thread_id"]
        )

        demo_results.append(
            {
                "id": item["id"],
                "mode": item["mode"],
                "question": item["question"],
                "answer": result["messages"][-1].content,
                "sources": result.get("sources", [])
            }
        )
else:
    print("Ejecución automática desactivada. Ejecuta las preguntas manualmente durante la demo.")

Ejecución automática desactivada. Ejecuta las preguntas manualmente durante la demo.


### 12.2. Guardado opcional de respuestas de demo

Si se ejecutan las preguntas automáticamente, esta celda permite guardar las respuestas en `outputs/demo_responses.md`.

In [24]:
from pathlib import Path

OUTPUTS_DIR = Path("outputs")
OUTPUTS_DIR.mkdir(exist_ok=True)

if demo_results:
    output_path = OUTPUTS_DIR / "demo_responses.md"

    lines = ["# Respuestas de demostración – AeroTwin AI\n"]

    for item in demo_results:
        lines.append(f"## Pregunta {item['id']} – {item['mode']}\n")
        lines.append(f"**Pregunta:** {item['question']}\n")
        lines.append("**Respuesta:**\n")
        lines.append(item["answer"])
        lines.append("\n**Fuentes recuperadas:**\n")

        for source in item["sources"]:
            lines.append(f"- {source}")

        lines.append("\n---\n")

    output_path.write_text("\n".join(lines), encoding="utf-8")

    print("Archivo generado:", output_path)
else:
    print("No hay resultados de demo guardados porque RUN_DEMO_AUTOMATICALLY está en False o no se han ejecutado preguntas.")

No hay resultados de demo guardados porque RUN_DEMO_AUTOMATICALLY está en False o no se han ejecutado preguntas.


## 13. Conclusiones y límites del MVP

AeroTwin AI demuestra cómo construir un asistente experto usando IA Generativa, RAG y agentes.

El MVP cumple los elementos principales del ejercicio:

- Usa documentos propios como base de conocimiento.
- Crea embeddings con Gemini.
- Construye una base vectorial con ChromaDB.
- Recupera contexto relevante mediante RAG.
- Genera respuestas con Gemini.
- Usa LangGraph para estructurar el agente.
- Incluye memoria básica de conversación.
- Permite interacción textual desde el notebook.
- Incluye preguntas documentadas para la demo.

### Límites actuales

- No consulta información de vuelos en tiempo real.
- No sustituye fuentes oficiales ni asesoramiento legal/aeronáutico.
- En modo demo se indexa una muestra de chunks para evitar problemas de cuota.
- La calidad de respuesta depende del contenido incluido en `data/raw/`.
- Para un despliegue real sería recomendable añadir actualización automática de fuentes, control de versiones documental y una interfaz web.

### Posibles mejoras futuras

- Interfaz con Streamlit.
- Clasificador automático de tipo de usuario.
- Búsqueda híbrida semántica + palabras clave.
- Filtros por aeropuerto, tipo de documento y audiencia.
- Evaluación sistemática de respuestas.
- Integración con fuentes actualizadas en tiempo real.

## 14. Checklist de entrega

Antes de entregar, comprobar:

- El repositorio contiene `README.md`, `requirements.txt`, `.env.example`, `data/raw/`, `outputs/demo_questions.md` y este notebook.
- El archivo `.env` no está subido a GitHub.
- `data/raw/` contiene los documentos oficiales.
- El notebook ejecuta correctamente al menos en modo demo.
- ChromaDB se crea desde el notebook.
- El agente responde con Gemini.
- Hay al menos cinco preguntas documentadas.
- La presentación oral muestra una demo breve y clara.